[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chung-I/MiRA_training_course_2026/blob/main/notebooks/llm_serving_demo.ipynb)

# Fast, efficient, VRAM-friendly LLM serving — live demo notebook

MiRA Training Course 2026 · 9/4 · 仲翊

Click the badge to open in **Google Colab**.

**Parts 1–5 need nothing at all** — no GPU, no installs, not even NumPy. They run on
the free CPU runtime. That is the whole argument of the talk: you can answer *"does it
fit, and how fast will it be"* before you touch a GPU.

**Parts 6–8 need a GPU.** In Colab: **Runtime → Change runtime type → T4 GPU** (free
tier). `torch` and `transformers` are pre-installed there, so there is nothing to
install. A T4 is a modest card, which is ideal here — the arithmetic in Parts 1–5
predicts its behaviour just as well as it predicts an H100's.

| # | Demo | Needs a GPU? |
|---|---|---|
| 1 | How much VRAM do the weights take? | no |
| 2 | The KV cache, per token and per user | no |
| 3 | How many users fit on your GPU? | no |
| 4 | The tokens/sec ceiling is memory bandwidth | no |
| 5 | Dollars per million tokens, and when to just use an API | no |
| 6 | Measure the real weight memory | yes |
| 7 | Measured tok/s vs. the predicted ceiling | yes |
| 8 | Batching is nearly free — measure it | yes |

---
## Part 1 — How much VRAM do the weights take?

`bytes = parameters × bytes per parameter`. That is the whole formula.

In [ ]:
GB = 1024**3

BYTES_PER_PARAM = {'fp32': 4, 'fp16': 2, 'bf16': 2, 'fp8': 1, 'int8': 1, 'int4': 0.5}

def weight_gb(params_billion, dtype='fp16'):
    return params_billion * 1e9 * BYTES_PER_PARAM[dtype] / GB

print(f'{"model":>10} | ' + ' | '.join(f'{d:>7}' for d in ['fp16','int8','int4']))
print('-' * 44)
for p in [0.5, 3, 7, 8, 32, 70, 405]:
    row = ' | '.join(f'{weight_gb(p, d):6.1f}G' for d in ['fp16','int8','int4'])
    print(f'{p:>8}B | {row}')

Read the 70B row out loud: **140 GB in fp16.** There is no single GPU you own that
holds that. At int4 it is 35 GB, which fits one 40 GB card — and that single fact
decides the entire architecture of the deployment.

---
## Part 2 — The KV cache, per token and per user

    KV bytes per token = 2 (K and V) × layers × kv_heads × head_dim × bytes_per_value

This is the number nobody computes, and it is the one that decides how many people can
use your server at once.

In [ ]:
def kv_bytes_per_token(layers, kv_heads, head_dim, bytes_per_value=2):
    return 2 * layers * kv_heads * head_dim * bytes_per_value

# Configs to sanity-check against the model's own config.json before you present.
MODELS = {
    #                    layers  kv_heads  head_dim   params(B)
    'Llama-2-7B  (MHA)': (32,      32,       128,       7),
    'Llama-3-8B  (GQA)': (32,       8,       128,       8),
    'Qwen2.5-7B  (GQA)': (28,       4,       128,       7),
    'Llama-3-70B (GQA)': (80,       8,       128,      70),
}

print(f'{"model":>19} | {"KV/token":>12} | {"4k ctx":>9} | {"32k ctx":>9}')
print('-' * 58)
for name, (L, kvh, hd, _) in MODELS.items():
    b = kv_bytes_per_token(L, kvh, hd)
    print(f'{name:>19} | {b/1024:8.0f} KiB | {b*4096/GB:7.2f} G | {b*32768/GB:7.2f} G')

Two things to say while this is on screen:

1. **Llama-2-7B costs 4× more cache per token than Llama-3-8B**, despite being a
   *smaller* model. The difference is Grouped-Query Attention: 32 KV heads versus 8.
   GQA is the reason modern models are servable at all.
2. **One user with a 32k conversation on Llama-3-8B costs 4 GB.** Your weights are a
   fixed cost paid once; the cache is paid *per user, per token*.

In [ ]:
# Optional: pull the real numbers straight from the model config instead of trusting the table.
try:
    from transformers import AutoConfig
    cfg = AutoConfig.from_pretrained('Qwen/Qwen2.5-7B-Instruct')  # ungated: works on Colab
    head_dim = getattr(cfg, 'head_dim', cfg.hidden_size // cfg.num_attention_heads)
    kv_heads = getattr(cfg, 'num_key_value_heads', cfg.num_attention_heads)
    b = kv_bytes_per_token(cfg.num_hidden_layers, kv_heads, head_dim)
    print(f'Qwen2.5-7B: layers={cfg.num_hidden_layers}  kv_heads={kv_heads}  head_dim={head_dim}')
    print(f'--> {b/1024:.0f} KiB per token, {b*8192/GB:.2f} GiB for an 8k conversation')
except Exception as e:
    print('Skipped (needs transformers + HF access):', type(e).__name__)
    print('Do this at home for whichever model you are actually deploying.')

---
## Part 3 — How many users fit on your GPU?

Whatever VRAM is left after the weights is divided among concurrent users. That is your
capacity, and quantization buys it back.

In [ ]:
def capacity(gpu_gb, params_b, layers, kv_heads, head_dim,
             ctx=8192, dtype='fp16', kv_dtype='fp16', overhead_gb=2.0):
    w = weight_gb(params_b, dtype)
    free = gpu_gb - w - overhead_gb
    per_user = kv_bytes_per_token(layers, kv_heads, head_dim,
                                  BYTES_PER_PARAM[kv_dtype]) * ctx / GB
    return w, max(free, 0.0), per_user, int(max(free, 0.0) // per_user)


L, KVH, HD, P = MODELS['Llama-3-8B  (GQA)']
for gpu_name, gpu_gb in [('RTX 4090 / 5090', 24), ('RTX 6000 Ada', 48), ('A100', 80)]:
    print(f'\n{gpu_name} ({gpu_gb} GB), Llama-3-8B, 8k context each:')
    for wd, kd in [('fp16','fp16'), ('int4','fp16'), ('int4','fp8')]:
        w, free, per_user, users = capacity(gpu_gb, P, L, KVH, HD, dtype=wd, kv_dtype=kd)
        print(f'   weights {wd:>4} + KV {kd:>4}: '
              f'{w:5.1f}G weights, {free:5.1f}G free, '
              f'{per_user:4.2f}G/user  -->  {users:3d} concurrent users')

**Same GPU. Same model. Two configuration choices. Several times the capacity.**

Notice that quantizing the *KV cache* helps exactly as much as quantizing the weights
once the weights are already small — and almost nobody does it.

---
## Part 4 — The tokens/sec ceiling is memory bandwidth

Generating one token requires reading **every weight** out of HBM. So:

    tokens/sec  ≲  memory bandwidth ÷ bytes of weights

This is an upper bound no amount of good code can beat.

In [ ]:
GPU_BW_TBS = {          # TB/s, from the spec sheet
    'Tesla T4': 0.32,           # <- the free Colab GPU
    'L4': 0.30,
    'RTX 4090': 1.01,
    'RTX 5090': 1.79,
    'A100-SXM4-40GB': 1.56,
    'A100 80GB': 2.04,
    'H100 SXM': 3.35,
}

def detect_bandwidth(default=0.32):
    '''Look up the bandwidth of whatever GPU this runtime happens to have.'''
    try:
        import torch
        if not torch.cuda.is_available():
            return None, default
        name = torch.cuda.get_device_name(0)
        for k, v in GPU_BW_TBS.items():
            if k.lower().replace('-', ' ') in name.lower().replace('-', ' '):
                return name, v
        return name, default
    except ImportError:
        return None, default

def tok_per_sec_ceiling(bw_tbs, params_b, dtype='fp16'):
    return bw_tbs * 1e12 / (params_b * 1e9 * BYTES_PER_PARAM[dtype])

print(f'{"GPU":>15} | {"8B fp16":>9} | {"8B int4":>9} | {"70B int4":>9}')
print('-' * 51)
for g, bw in GPU_BW_TBS.items():
    print(f'{g:>15} | {tok_per_sec_ceiling(bw, 8):8.0f}/s | '
          f'{tok_per_sec_ceiling(bw, 8, "int4"):8.0f}/s | '
          f'{tok_per_sec_ceiling(bw, 70, "int4"):8.0f}/s')

name, bw = detect_bandwidth()
print()
print(f'this runtime: {name or "no GPU"}  ->  assuming {bw} TB/s')

**Why quantization makes decoding faster** is now obvious, and it is not the reason
most people give: int4 does not do less arithmetic than fp16 in any interesting sense.
It moves **one quarter of the bytes**, and moving bytes is the bottleneck.

Also worth saying: a human reads at roughly 5–10 tokens/sec. Anything above ~30 tok/s
per user is invisible to a reader — so spend the surplus on more users, not on speed.

---
## Part 5 — Dollars per million tokens

    $ per 1M tokens = GPU $/hour ÷ (tokens/sec × 3600) × 1,000,000

The catch is `tokens/sec` must be your *aggregate* throughput across all concurrent
requests, not single-stream speed. This is why batching is an economic decision.

In [ ]:
def cost_per_million(gpu_dollars_per_hour, aggregate_tok_per_sec):
    return gpu_dollars_per_hour / (aggregate_tok_per_sec * 3600) * 1e6

GPU_HOURLY = 2.00       # a rented A100-ish, adjust to your own numbers

print(f'{"utilisation":>28} | {"$ / 1M tokens":>14}')
print('-' * 46)
for label, tps in [('1 user, single stream', 60),
                   ('8 concurrent', 400),
                   ('32 concurrent', 1200),
                   ('well-tuned vLLM, batched', 2500)]:
    print(f'{label:>28} | {cost_per_million(GPU_HOURLY, tps):13.2f}')

print()
print('An owned GPU sitting idle costs the same as a busy one.')
print('Self-hosting wins on sustained utilisation, a fine-tuned model,')
print('or data that legally cannot leave the building. Otherwise, use an API.')

---
## Part 6 — Measure the real weight memory (needs a GPU)

Everything above was arithmetic. Now check it. From here on you need `torch`,
`transformers`, and a GPU. A small model is fine — the *point* is that the prediction
matches.

In [ ]:
import importlib.util
HAVE_TORCH = importlib.util.find_spec('torch') is not None
GPU = False
if HAVE_TORCH:
    import torch
    GPU = torch.cuda.is_available()
    if GPU:
        p = torch.cuda.get_device_properties(0)
        print(f'{p.name}, {p.total_memory/GB:.1f} GB VRAM')
    else:
        print('torch is installed but no GPU is visible - parts 6-8 will be skipped')
else:
    print('torch not installed - parts 6-8 will be skipped')

In [ ]:
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'      # small on purpose; swap for anything you like

if GPU:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    before = torch.cuda.memory_allocated()

    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).cuda().eval()

    n_params = sum(p.numel() for p in model.parameters())
    measured = (torch.cuda.memory_allocated() - before) / GB
    predicted = n_params * 2 / GB
    print(f'parameters        : {n_params/1e9:.3f} B')
    print(f'predicted (fp16)  : {predicted:.3f} GB')
    print(f'measured on GPU   : {measured:.3f} GB')
    print(f'ratio             : {measured/predicted:.3f}   (should be very close to 1.0)')

---
## Part 7 — Measured tok/s vs. the predicted ceiling

The most useful cell in this notebook. If the measurement lands somewhere near the
prediction, you have just proved that decoding is bandwidth-bound.

In [ ]:
import time

def measure_decode(model, tok, prompt='Explain what a robot is, in detail.', new_tokens=128):
    ids = tok(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():                                   # warm up
        model.generate(**ids, max_new_tokens=8, do_sample=False)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=new_tokens, do_sample=False)
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    generated = out.shape[1] - ids['input_ids'].shape[1]
    return generated / dt


if GPU:
    measured_tps = measure_decode(model, tok)
    gpu_name, my_bw = detect_bandwidth()
    ceiling = tok_per_sec_ceiling(my_bw, n_params/1e9, 'fp16')
    print(f'{gpu_name} at {my_bw} TB/s')
    print(f'measured decode      : {measured_tps:7.1f} tok/s')
    print(f'bandwidth ceiling    : {ceiling:7.1f} tok/s')
    print(f'efficiency           : {measured_tps/ceiling*100:6.1f}% of the theoretical roof')
    print()
    print('HuggingFace generate() typically reaches a modest fraction of the roof.')
    print('That gap is exactly what vLLM and TensorRT-LLM exist to close.')

---
## Part 8 — Batching is nearly free: measure it

Weights are read once for the whole batch. So going from 1 request to 16 should
multiply throughput without multiplying latency. This is the single most important
economic fact about serving.

In [ ]:
def measure_batch(model, tok, batch_size, new_tokens=64):
    prompts = ['Explain what a robot is, in detail.'] * batch_size
    tok.pad_token = tok.pad_token or tok.eos_token
    ids = tok(prompts, return_tensors='pt', padding=True).to('cuda')
    with torch.no_grad():
        model.generate(**ids, max_new_tokens=4, do_sample=False)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        model.generate(**ids, max_new_tokens=new_tokens, do_sample=False)
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    return batch_size * new_tokens / dt, dt


if GPU:
    print(f'{"batch":>6} | {"total tok/s":>12} | {"per-user tok/s":>15} | {"wall time":>10}')
    print('-' * 52)
    base = None
    for bs in (1, 2, 4, 8, 16):
        try:
            total, dt = measure_batch(model, tok, bs)
            base = base or total
            print(f'{bs:>6} | {total:11.1f}  | {total/bs:14.1f}  | {dt:9.2f}s')
        except torch.cuda.OutOfMemoryError:
            print(f'{bs:>6} | out of memory  <- this is the KV cache from Part 2')
            break
    print()
    print('Total throughput climbs steeply; per-user speed falls only slowly.')
    print('That gap is where all the serving economics live.')

---
## What to take away

1. **Weights are a fixed cost, the KV cache is a per-user cost.** Compute both before
   you download anything (Parts 1–3).
2. **Decode speed is bandwidth ÷ bytes.** Quantization is fast because it moves fewer
   bytes, not because it does less math (Parts 4 and 7).
3. **Batching converts spare bandwidth into users, almost for free** (Part 8) — and the
   KV cache is what limits the batch, which closes the loop back to Part 2.
4. Report **TTFT, inter-token latency, and throughput at a stated concurrency**, or
   report nothing.

Next step in real life: none of this notebook is how you actually serve a model. Use
**vLLM** or **SGLang**, which do continuous batching and PagedAttention for you:

    pip install vllm
    vllm serve Qwen/Qwen2.5-7B-Instruct --quantization awq --max-model-len 8192

Then re-run the arithmetic from Parts 1–5 against what the server reports. It should
line up.